In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier

df = pd.read_csv('../data/accepted_2007_to_2018Q4.csv.gz', low_memory=False)

cols = ['loan_amnt', 'term', 'int_rate', 'grade', 'sub_grade', 'emp_length',
        'home_ownership', 'annual_inc', 'purpose', 'dti', 'loan_status']
df = df[cols]
df = df[df['loan_status'].isin(['Fully Paid', 'Charged Off'])].copy()
df['default'] = (df['loan_status'] == 'Charged Off').astype(int)

print(f"Total: {len(df):,}")

Total: 1,345,310


In [2]:
# Feature 1: indicador de valor faltante en emp_length (ANTES de convertirlo a número)
df['emp_length_missing'] = df['emp_length'].isnull().astype(int)

# Ahora sí, convertimos emp_length a número (igual que antes)
emp_map = {'< 1 year': 0, '1 year': 1, '2 years': 2, '3 years': 3, '4 years': 4,
           '5 years': 5, '6 years': 6, '7 years': 7, '8 years': 8, '9 years': 9, '10+ years': 10}
df['emp_length'] = df['emp_length'].map(emp_map)
df['emp_length'] = df['emp_length'].fillna(df['emp_length'].median())

# Feature 2: ratio préstamo / ingreso anual
df['loan_to_income'] = df['loan_amnt'] / df['annual_inc']

# Feature 3: sub_grade en vez de grade, convertido a orden numérico
subgrades = sorted(df['sub_grade'].dropna().unique())
subgrade_map = {sg: i+1 for i, sg in enumerate(subgrades)}
df['sub_grade_num'] = df['sub_grade'].map(subgrade_map)

df[['emp_length_missing', 'loan_to_income', 'sub_grade', 'sub_grade_num']].head()

,emp_length_missing,loan_to_income,sub_grade,sub_grade_num
0,0,0.065455,C4,14
1,0,0.380000,C1,11
2,0,0.317460,B4,9
4,0,0.099585,F1,26
5,0,0.351471,C3,13


In [3]:
features_mejoradas = ['loan_amnt', 'term', 'int_rate', 'sub_grade_num', 'emp_length',
                       'emp_length_missing', 'loan_to_income', 'home_ownership',
                       'annual_inc', 'purpose', 'dti']

X = df[features_mejoradas].copy()
y = df['default']

X['term'] = X['term'].str.extract(r'(\d+)').astype(float)
X = pd.get_dummies(X, columns=['home_ownership', 'purpose'], drop_first=True)
X['dti'] = X['dti'].fillna(X['dti'].median())

print(f"Shape final: {X.shape}")
X.isnull().sum().sum()  # debería dar 0

Shape final: (1345310, 27)


np.int64(0)

In [7]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

modelo_mejorado = XGBClassifier(n_estimators=200, max_depth=4, learning_rate=0.1, random_state=42)
modelo_mejorado.fit(X_train, y_train)

proba_pred = modelo_mejorado.predict_proba(X_test)[:, 1]
auc_mejorado = roc_auc_score(y_test, proba_pred)

print(f"AUC-ROC modelo original (grade + variables base): 0.7120")
print(f"AUC-ROC modelo mejorado (sub_grade + nuevas features): {auc_mejorado:.4f}")
print(f"Mejora: {(auc_mejorado - 0.7120)*100:.2f} puntos")

AUC-ROC modelo original (grade + variables base): 0.7120
AUC-ROC modelo mejorado (sub_grade + nuevas features): 0.7147
Mejora: 0.27 puntos


In [8]:
importancias_v2 = pd.Series(modelo_mejorado.feature_importances_, index=X.columns).sort_values(ascending=False)
importancias_v2.head(10)

sub_grade_num              0.504380
term                       0.095275
home_ownership_MORTGAGE    0.086248
home_ownership_RENT        0.061689
emp_length_missing         0.059172
loan_to_income             0.042860
dti                        0.026767
purpose_small_business     0.019836
int_rate                   0.016243
purpose_wedding            0.010536
dtype: float32

In [5]:
# Buscamos si hay valores infinitos o inválidos
print("Valores infinitos en X:", np.isinf(X.select_dtypes(include=[np.number])).sum().sum())
print("Préstamos con annual_inc = 0:", (df['annual_inc'] == 0).sum())

Valores infinitos en X: 361
Préstamos con annual_inc = 0: 361


In [6]:
# Reemplazamos los infinitos por un valor alto pero finito (el máximo válido * 2, por ejemplo)
max_valido = X.loc[~np.isinf(X['loan_to_income']), 'loan_to_income'].max()
X['loan_to_income'] = X['loan_to_income'].replace([np.inf, -np.inf], max_valido * 2)

# Verificamos que ya no queden infinitos
print("Valores infinitos después de la corrección:", np.isinf(X.select_dtypes(include=[np.number])).sum().sum())

Valores infinitos después de la corrección: 0
